# Data Audit and Sampling

## Goal
Inspect the raw files, build the balanced subset, and verify split integrity.

## Required checks
- row counts
- missing values
- rating distribution
- category balance
- `parent_asin` split leakage check


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.data import process_data, split_by_asin, clean_text
import pandas as pd

In [ ]:
# This will download the GZ files, sample 500,000 strings from each,
# select necessary columns, assign sentiment labels, and save to parquet.
df = process_data(
    data_dir=str(PROJECT_ROOT / "data"),
    n_samples=500000,
    random_seed=42
)

print("Total dataset shape:", df.shape)
print(df['main_category'].value_counts())

In [ ]:
# Check sentiment class distribution
print(df['sentiment'].value_counts(normalize=True))

In [ ]:
# Clean text applying lightweight tokenization
df['cleaned_text'] = df['text'].astype(str).apply(clean_text)
df[['text', 'cleaned_text']].head(3)

In [ ]:
# Split into train/validation/test protecting parent_asin
train_df, val_df, test_df = split_by_asin(df, test_size=0.15, val_size=0.15)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# Ensure no leakage between train and test
train_asins = set(train_df['parent_asin'].unique())
test_asins = set(test_df['parent_asin'].unique())
leakage = train_asins.intersection(test_asins)

print(f"Leaked ASINs between Train and Test: {len(leakage)}")

In [ ]:
# Save processed splits
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

train_df.to_parquet(processed_dir / "train.parquet")
val_df.to_parquet(processed_dir / "val.parquet")
test_df.to_parquet(processed_dir / "test.parquet")
print("Saved splits to data/processed/")
